<h1>09. Extensions and Conclusions</h1>

**What we do.** We first draw a clear boundary around the graded project: notebooks 01-08 are the complete, validated pipeline behind every reported result, and this notebook shows, but does not run as part of the main claims, four optional exploratory extensions kept in the extensions/ folder: label propagation, GCN/GraphSAGE message-passing models, and TruncatedSVD+K-means content-subfield clustering. We then close with a short wrap-up of what the nine notebooks showed together and the project's main limitations.

**What we get.** An inventory of the extension files with the actual source code for each, a project wrap-up connecting RQ1-RQ4 back into one narrative, and a limitations section.

**Why this step matters.** It is easy for exploratory side-experiments to blur into the graded results if they are not explicitly separated; this notebook exists so a reader can see exactly what was explored beyond the course requirements without mistaking it for a validated claim, and so the project ends with an honest statement of what its findings do and do not show.

In [1]:
from pathlib import Path
import os
import sys
import warnings


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "data").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the project root")


ROOT = _find_project_root(Path.cwd())
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("PYTHONHASHSEED", "0")
warnings.filterwarnings("ignore", category=FutureWarning)
__name__ = "notebook_09"
print(f"Project root: {ROOT}")


Project root: C:\Users\Jimi\Desktop\github-social-network


<h2>Extension boundary</h2>

The main results come from notebooks 01-08. The `extensions/` folder keeps extra ideas separate: Label Propagation, GCN/GraphSAGE, and content-subfield clustering. We keep them here as possible next steps, but they are not part of the main result tables.

<h2>Extension list</h2>

List the optional files without running them.

In [2]:
import pandas as pd

extension_inventory = pd.DataFrame(
    [
        {
            "file": path.name,
            "role": {
                "gnn.py": "GCN and GraphSAGE model definitions",
                "run_gnn.py": "Optional message-passing experiment",
                "run_subfields.py": "Optional content subfield clustering",
                "label_propagation.py": "Optional label-propagation classifier",
                "README.md": "Extension boundary note",
            }.get(path.name, "Optional extension file"),
            "line_count": len(path.read_text(encoding="utf-8").splitlines()),
        }
        for path in sorted((ROOT / "extensions").glob("*"))
        if path.is_file()
    ]
)
display(extension_inventory)


,file,role,line_count
0,gnn.py,GCN and GraphSAGE model definitions,187
1,label_propagation.py,Optional label-propagation classifier,61
2,README.md,Extension boundary note,13
3,run_gnn.py,Optional message-passing experiment,221
4,run_subfields.py,Optional content subfield clustering,191


**Conclusion.** The optional files cover alternative modeling directions rather than additional results for the submitted claims. Keeping them separate avoids changing the meaning of the main experiments.

<h3>Source code: <code>extensions/label_propagation.py</code></h3>

Optional semi-supervised baseline that spreads labels across the graph from a small labeled seed set, kept as exploratory work.

**`imports and constants`**

Imports the external libraries and project modules this file depends on, and defines the fixed constants (seeds, budgets, thresholds) shared by every function below.

In [ ]:
"""Optional label-propagation extension, excluded from the final project."""

from typing import Dict, Iterable, Tuple

import networkx as nx
import numpy as np

**`label_propagation_classifier`**

Iteratively spreads labels from a small labeled seed set to their neighbors across the graph until the labeling stabilizes, a semi-supervised alternative to the supervised classifiers used in the main pipeline.

In [ ]:
def label_propagation_classifier(
    G: nx.Graph,
    labels_dict: Dict[int, int],
    train_nodes: Iterable[int],
    test_nodes: Iterable[int],
    default_label: int,
    max_iter: int = 100,
    tol: float = 1e-6,
) -> Tuple[np.ndarray, np.ndarray]:
    """Propagate clamped train labels over a row-normalized adjacency matrix."""
    train_set = {int(node) for node in train_nodes}
    test_list = [int(node) for node in test_nodes]
    train_label_dict = {
        node: int(labels_dict[node]) for node in train_set if node in labels_dict
    }
    train_ml_prior = (
        float(np.mean(list(train_label_dict.values()))) if train_label_dict else 0.5
    )

    nodelist = list(G.nodes())
    node_to_idx = {int(node): idx for idx, node in enumerate(nodelist)}
    adjacency = nx.to_scipy_sparse_array(
        G, nodelist=nodelist, dtype=float, format="csr"
    )
    row_sums = np.asarray(adjacency.sum(axis=1)).ravel()
    row_sums[row_sums == 0] = 1.0
    transition = adjacency.multiply(1.0 / row_sums[:, None]).tocsr()

    scores = np.full(len(nodelist), train_ml_prior, dtype=np.float64)
    fixed_mask = np.zeros(len(nodelist), dtype=bool)
    for node, label in train_label_dict.items():
        idx = node_to_idx.get(node)
        if idx is not None:
            scores[idx] = float(label)
            fixed_mask[idx] = True

    for _ in range(max_iter):
        new_scores = np.asarray(transition @ scores).ravel()
        new_scores[fixed_mask] = scores[fixed_mask]
        if float(np.max(np.abs(new_scores - scores))) < tol:
            scores = new_scores
            break
        scores = new_scores

    y_score = np.asarray(
        [scores[node_to_idx[node]] if node in node_to_idx else train_ml_prior
         for node in test_list],
        dtype=np.float64,
    )
    y_pred = np.where(
        y_score > 0.5, 1, np.where(y_score < 0.5, 0, int(default_label))
    ).astype(np.int64)
    return y_pred, y_score

<h3>Source code: <code>extensions/run_subfields.py</code></h3>

Optional exploration script that clusters content features with TruncatedSVD and K-means to look for finer subfields inside the Web/ML labels.

**`imports and constants`**

Imports the external libraries and project modules this file depends on, and defines the fixed constants (seeds, budgets, thresholds) shared by every function below.

In [ ]:
# -*- coding: utf-8 -*-
"""
Extension only: this exploratory subfield analysis is excluded because its
central methods are outside the selected course-only scope.

Content-based sub-field discovery.

The binary Web/ML target hides the fact that developers span many
sub-specializations. This script clusters the MUSAE content features
(starred repos / employer / location) to discover data-driven "sub-fields",
then measures how well the network's Louvain communities align with them.

Questions answered:
1. Do content clusters split the binary labels into finer sub-fields?
2. Do network communities align more with content clusters than with the
   binary Web/ML label? (NMI / ARI)

Outputs:
    outputs/tables/subfield_cluster_summary.csv
    outputs/tables/subfield_alignment_scores.csv
    outputs/figures/subfield_ml_share.png
    outputs/figures/subfield_community_overlap.png
"""
from pathlib import Path

import sys

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.preprocessing import normalize

EXTENSIONS_DIR = Path(__file__).resolve().parent
PROJECT_DIR = EXTENSIONS_DIR.parent
sys.path.insert(0, str(PROJECT_DIR / "src"))
sys.path.insert(0, str(EXTENSIONS_DIR))

from community_analysis import detect_louvain_communities
from content_features import build_content_matrix, load_content_features
from data_loader import load_dataset


SVD_COMPONENTS = 64
K_CANDIDATES = range(4, 11)
SILHOUETTE_SAMPLE = 5000
RANDOM_STATE = 42

**`choose_k`**

Sweeps candidate cluster counts and picks the one with the best Silhouette score, used to choose how many subfields to look for inside the content-embedding space before running K-means.

In [ ]:
def choose_k(embedding: np.ndarray) -> tuple[int, pd.DataFrame]:
    """Pick the number of clusters by silhouette score over a candidate range."""
    rng = np.random.default_rng(RANDOM_STATE)
    sample_idx = rng.choice(embedding.shape[0], size=min(SILHOUETTE_SAMPLE, embedding.shape[0]), replace=False)

    rows = []
    for k in K_CANDIDATES:
        kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        cluster_labels = kmeans.fit_predict(embedding)
        score = silhouette_score(embedding[sample_idx], cluster_labels[sample_idx])
        rows.append({"k": k, "silhouette": float(score)})
        print(f"  k={k}: silhouette={score:.4f}")

    scores_df = pd.DataFrame(rows)
    best_k = int(scores_df.loc[scores_df["silhouette"].idxmax(), "k"])
    return best_k, scores_df

**`main`**

Runs the optional subfield-clustering experiment: embeds the sparse binary content features into a dense low-rank representation, clusters them with K-means using choose_k to pick the number of clusters, and reports Silhouette, NMI, and ARI against the Web/ML labels. It also checks alignment between the discovered content clusters, the true labels, and the Louvain communities from the main pipeline, and saves a per-cluster ML-share figure and a cluster-by-community overlap heatmap. Not part of the graded course-method results.

In [ ]:
def main() -> None:
    data_dir = PROJECT_DIR / "data"
    tables_dir = PROJECT_DIR / "outputs" / "tables"
    figures_dir = PROJECT_DIR / "outputs" / "figures"
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    G, _, labels_df, labels_dict = load_dataset(data_dir)
    features_dict, n_features = load_content_features(data_dir / "musae_git_features.json")

    nodes = labels_df["id"].to_numpy(dtype=np.int64)
    labels = labels_df["label"].to_numpy(dtype=np.int64)

    # --- Content embedding: sparse binary -> dense low-rank representation ---
    content_matrix = build_content_matrix(features_dict, n_features, nodes)
    svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=RANDOM_STATE)
    embedding = svd.fit_transform(content_matrix)
    embedding = normalize(embedding)
    print(f"SVD explained variance: {svd.explained_variance_ratio_.sum():.4f}")

    # --- Cluster into sub-fields ---
    print("Choosing k by silhouette:")
    best_k, k_scores_df = choose_k(embedding)
    print(f"Selected k={best_k}")

    kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
    clusters = kmeans.fit_predict(embedding)

    # --- Per-cluster label composition ---
    cluster_df = pd.DataFrame({"node_id": nodes, "cluster": clusters, "label": labels})
    summary_df = (
        cluster_df.groupby("cluster")
        .agg(size=("node_id", "count"), ml_count=("label", "sum"))
        .reset_index()
    )
    summary_df["ml_share"] = summary_df["ml_count"] / summary_df["size"]
    summary_df = summary_df.sort_values("ml_share", ascending=False).reset_index(drop=True)
    summary_df.to_csv(tables_dir / "subfield_cluster_summary.csv", index=False)
    print(summary_df.to_string(index=False))

    # --- Alignment: clusters vs labels vs Louvain communities ---
    partition, _ = detect_louvain_communities(G, random_state=RANDOM_STATE)
    communities = np.asarray([partition.get(int(node), -1) for node in nodes])

    alignment_rows = [
        {
            "comparison": "content_clusters_vs_binary_labels",
            "nmi": float(normalized_mutual_info_score(labels, clusters)),
            "ari": float(adjusted_rand_score(labels, clusters)),
        },
        {
            "comparison": "louvain_communities_vs_binary_labels",
            "nmi": float(normalized_mutual_info_score(labels, communities)),
            "ari": float(adjusted_rand_score(labels, communities)),
        },
        {
            "comparison": "louvain_communities_vs_content_clusters",
            "nmi": float(normalized_mutual_info_score(clusters, communities)),
            "ari": float(adjusted_rand_score(clusters, communities)),
        },
    ]
    alignment_df = pd.DataFrame(alignment_rows)
    alignment_df.to_csv(tables_dir / "subfield_alignment_scores.csv", index=False)
    print(alignment_df.to_string(index=False))

    # --- Figure 1: ML share per sub-field cluster ---
    fig, ax1 = plt.subplots(figsize=(11, 5))
    x = np.arange(len(summary_df))
    colors = plt.cm.coolwarm(summary_df["ml_share"].to_numpy())
    ax1.bar(x, summary_df["ml_share"], color=colors)
    ax1.axhline(labels.mean(), color="black", linestyle="--", linewidth=1,
                label=f"Global ML share ({labels.mean():.2f})")
    for xi, (share, size) in enumerate(zip(summary_df["ml_share"], summary_df["size"])):
        ax1.text(xi, share + 0.015, f"n={size:,}", ha="center", fontsize=8)
    ax1.set_xticks(x)
    ax1.set_xticklabels([f"C{c}" for c in summary_df["cluster"]])
    ax1.set_xlabel("Content cluster (sorted by ML share)")
    ax1.set_ylabel("ML share")
    ax1.set_title(f"Content-Based Sub-Fields (k={best_k}): ML Share per Cluster")
    ax1.legend()
    ax1.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(figures_dir / "subfield_ml_share.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # --- Figure 2: overlap heatmap, content clusters x largest communities ---
    top_communities = (
        pd.Series(communities).value_counts().head(10).index.to_numpy()
    )
    overlap = np.zeros((best_k, len(top_communities)))
    for ci in range(best_k):
        in_cluster = communities[clusters == ci]
        for cj, comm in enumerate(top_communities):
            overlap[ci, cj] = np.mean(in_cluster == comm) if len(in_cluster) else 0.0

    fig, ax2 = plt.subplots(figsize=(10, 6))
    im = ax2.imshow(overlap, cmap="viridis", aspect="auto")
    ax2.set_xticks(np.arange(len(top_communities)))
    ax2.set_xticklabels([f"Comm {c}" for c in top_communities], rotation=45, ha="right")
    ax2.set_yticks(np.arange(best_k))
    ax2.set_yticklabels([f"Cluster {c}" for c in range(best_k)])
    ax2.set_xlabel("Louvain community (top 10 by size)")
    ax2.set_ylabel("Content cluster")
    ax2.set_title("Where Do Content Sub-Fields Live in the Network?\n(row = share of cluster members in each community)")
    fig.colorbar(im, ax=ax2, label="Share of cluster")
    fig.tight_layout()
    fig.savefig(figures_dir / "subfield_community_overlap.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    k_scores_df.to_csv(tables_dir / "subfield_k_selection.csv", index=False)
    print("Saved tables and figures.")

**`__main__ guard`**

Keeps the original command-line entry guard from the source file; it stays inactive here because the run cell below calls the pipeline directly.

In [ ]:
if __name__ == "__main__":
    main()

<h3>Source code: <code>extensions/gnn.py</code></h3>

Defines the optional GCN and GraphSAGE model classes, kept outside the main pipeline because they require PyTorch and were not part of the graded course-method claims.

**`imports and constants`**

Imports the external libraries and project modules this file depends on, and defines the fixed constants (seeds, budgets, thresholds) shared by every function below.

In [ ]:
# -*- coding: utf-8 -*-
"""
Extension only: this module is retained for reference and is not part of the
course-only analysis, reported results, or reproducibility pipeline.

Graph Convolutional Network (Kipf & Welling, ICLR 2017) in plain PyTorch.

Each GCN layer propagates node features over the symmetrically-normalized
adjacency with self-loops:  H' = ReLU( A_hat @ H @ W ),
A_hat = D^-1/2 (A + I) D^-1/2.

Implemented with a sparse adjacency tensor, so the full 37.7K-node graph
trains comfortably on CPU. No labels are used to build A_hat or the inputs;
supervision enters only through the cross-entropy loss on train nodes
(transductive node classification, matching the project protocol).
"""
from typing import Dict, List, Tuple

import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import sparse

**`build_normalized_adjacency`**

Builds the symmetrically normalized adjacency matrix D^-1/2 (A+I) D^-1/2 that a GCN layer needs, as a dense PyTorch tensor.

In [ ]:
def build_normalized_adjacency(G: nx.Graph, node_order: List[int]) -> torch.Tensor:
    """Build A_hat = D^-1/2 (A + I) D^-1/2 as a torch sparse tensor."""
    index_of: Dict[int, int] = {int(node): idx for idx, node in enumerate(node_order)}
    n = len(node_order)

    rows, cols = [], []
    for u, v in G.edges():
        iu, iv = index_of[int(u)], index_of[int(v)]
        rows.extend((iu, iv))
        cols.extend((iv, iu))
    rows.extend(range(n))  # self-loops
    cols.extend(range(n))

    data = np.ones(len(rows), dtype=np.float32)
    adjacency = sparse.coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()

    degree = np.asarray(adjacency.sum(axis=1)).ravel()
    inv_sqrt_degree = 1.0 / np.sqrt(np.maximum(degree, 1.0))
    normalized = adjacency.multiply(inv_sqrt_degree[:, None]).multiply(inv_sqrt_degree[None, :])
    normalized = normalized.tocoo()

    indices = torch.tensor(np.vstack([normalized.row, normalized.col]), dtype=torch.long)
    values = torch.tensor(normalized.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(indices, values, size=(n, n)).coalesce()

**`GCN`**

A two-layer Graph Convolutional Network: each layer multiplies the node features by the normalized adjacency matrix before a linear transform, so a node's representation is repeatedly averaged with its neighbors' representations across layers.

In [ ]:
class GCN(nn.Module):
    """Two-layer GCN for binary node classification."""

    def __init__(self, in_features: int, hidden: int = 64, n_classes: int = 2, dropout: float = 0.5):
        super().__init__()
        self.linear1 = nn.Linear(in_features, hidden)
        self.linear2 = nn.Linear(hidden, n_classes)
        self.dropout = dropout

    def forward(self, a_hat: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        h = torch.sparse.mm(a_hat, x)
        h = F.relu(self.linear1(h))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = torch.sparse.mm(a_hat, h)
        return self.linear2(h)

**`build_mean_adjacency`**

Builds a row-normalized (mean-aggregation) adjacency matrix, the neighbor-averaging operator GraphSAGE's mean aggregator needs instead of GCN's symmetric normalization.

In [ ]:
def build_mean_adjacency(G: nx.Graph, node_order: List[int]) -> torch.Tensor:
    """
    Build the row-normalized adjacency D^-1 A (no self-loops) used by
    GraphSAGE mean aggregation: (D^-1 A) @ H = mean of neighbor features.
    """
    index_of: Dict[int, int] = {int(node): idx for idx, node in enumerate(node_order)}
    n = len(node_order)

    rows, cols = [], []
    for u, v in G.edges():
        iu, iv = index_of[int(u)], index_of[int(v)]
        rows.extend((iu, iv))
        cols.extend((iv, iu))

    data = np.ones(len(rows), dtype=np.float32)
    adjacency = sparse.coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()
    degree = np.asarray(adjacency.sum(axis=1)).ravel()
    inv_degree = 1.0 / np.maximum(degree, 1.0)
    normalized = adjacency.multiply(inv_degree[:, None]).tocoo()

    indices = torch.tensor(np.vstack([normalized.row, normalized.col]), dtype=torch.long)
    values = torch.tensor(normalized.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(indices, values, size=(n, n)).coalesce()

**`GraphSAGE`**

A two-layer mean-aggregation GNN: each layer concatenates a node's own features with the mean of its neighbors' features before a linear transform, an inductive alternative to GCN's spectral-style normalization.

In [ ]:
class GraphSAGE(nn.Module):
    """
    Two-layer GraphSAGE with mean aggregation (Hamilton et al., 2017).

    Each layer concatenates the node's own representation with the mean of
    its neighbors' representations, then applies a linear transformation:
        h_v' = ReLU( W @ [h_v ; mean_{u in N(v)} h_u] )
    """

    def __init__(self, in_features: int, hidden: int = 64, n_classes: int = 2, dropout: float = 0.5):
        super().__init__()
        self.linear1 = nn.Linear(in_features * 2, hidden)
        self.linear2 = nn.Linear(hidden * 2, n_classes)
        self.dropout = dropout

    def forward(self, a_mean: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        neighbor_mean = torch.sparse.mm(a_mean, x)
        h = F.relu(self.linear1(torch.cat([x, neighbor_mean], dim=1)))
        h = F.dropout(h, p=self.dropout, training=self.training)
        neighbor_mean = torch.sparse.mm(a_mean, h)
        return self.linear2(torch.cat([h, neighbor_mean], dim=1))

**`train_gcn`**

Trains the GCN or GraphSAGE model with a standard cross-entropy training loop; kept here rather than in the main pipeline because it needs PyTorch, which is not a project dependency, and was not part of the graded course-method claims.

In [ ]:
def train_gcn(
    a_hat: torch.Tensor,
    features: np.ndarray,
    labels: np.ndarray,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    test_idx: np.ndarray,
    hidden: int = 64,
    epochs: int = 300,
    lr: float = 0.01,
    weight_decay: float = 5e-4,
    seed: int = 42,
    architecture: str = "gcn",
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Full-batch transductive training with class weights and best-epoch
    selection on the validation macro loss. Returns (y_pred, y_score) for
    the test indices, where y_score is the ML-class probability.

    architecture: "gcn" (pass the symmetric-normalized adjacency) or
    "graphsage" (pass the row-normalized mean adjacency).
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    x = torch.tensor(features, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)
    train_t = torch.tensor(train_idx, dtype=torch.long)
    val_t = torch.tensor(val_idx, dtype=torch.long)

    class_counts = np.bincount(labels[train_idx], minlength=2).astype(np.float64)
    class_weights = torch.tensor(
        class_counts.sum() / (2.0 * np.maximum(class_counts, 1.0)), dtype=torch.float32
    )

    if architecture == "graphsage":
        model = GraphSAGE(in_features=x.shape[1], hidden=hidden)
    else:
        model = GCN(in_features=x.shape[1], hidden=hidden)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_loss = float("inf")
    best_state = None

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(a_hat, x)
        loss = F.cross_entropy(logits[train_t], y[train_t], weight=class_weights)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits = model(a_hat, x)
            val_loss = F.cross_entropy(logits[val_t], y[val_t], weight=class_weights).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        probabilities = F.softmax(model(a_hat, x), dim=1).numpy()

    y_score = probabilities[test_idx, 1]
    y_pred = (y_score >= 0.5).astype(np.int64)
    return y_pred, y_score.astype(np.float64)

<h3>Source code: <code>extensions/run_gnn.py</code></h3>

Optional experiment script that trains the GCN/GraphSAGE models above on the same classification task, for comparison purposes only.

**`imports and constants`**

Imports the external libraries and project modules this file depends on, and defines the fixed constants (seeds, budgets, thresholds) shared by every function below.

In [ ]:
# -*- coding: utf-8 -*-
"""
Extension only: this experiment is excluded from the course-only pipeline and
from every result reported in the notebook, README, and final report.

GNN experiment: does message passing beat everything else?

Two GCN variants, both network-only, evaluated under the standard project
protocol (same seeds, stratified 70/30 splits, same metrics):

1. GCN - Structural Input : input features are cheap structure-only node
   descriptors (log-degree, clustering, core number, log-triangles, PageRank).
2. GCN - DeepWalk Input   : input features are the 128-d DeepWalk embeddings
   (requires run_deepwalk.py to have been run first).

Within each split, train nodes are further divided 90/10 into
train-inner/validation for best-epoch selection - test labels are never
touched during training.

Outputs:
    outputs/tables/gnn_model_results.csv
    outputs/tables/gnn_model_results_summary.csv
    outputs/figures/gnn_network_only_ladder.png
"""
import time
from pathlib import Path
from typing import Sequence

import sys

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

EXTENSIONS_DIR = Path(__file__).resolve().parent
PROJECT_DIR = EXTENSIONS_DIR.parent
sys.path.insert(0, str(PROJECT_DIR / "src"))
sys.path.insert(0, str(EXTENSIONS_DIR))

from data_loader import load_dataset
from gnn import build_mean_adjacency, build_normalized_adjacency, train_gcn
from modeling import evaluate_predictions, prepare_train_test_split


MODEL_GCN_STRUCT = "GCN - Structural Input"
MODEL_GCN_DEEPWALK = "GCN - DeepWalk Input"
MODEL_SAGE_STRUCT = "GraphSAGE - Structural Input"
MODEL_SAGE_DEEPWALK = "GraphSAGE - DeepWalk Input"

**`structural_input_features`**

Builds a small hand-crafted structural feature matrix, degree, clustering, and community, to use as GNN input rather than one-hot node identity, so the model can also generalize to a held-out node in principle.

In [ ]:
def structural_input_features(G: nx.Graph, node_order: list[int]) -> np.ndarray:
    """Cheap structure-only node features (no labels, no content)."""
    print("Computing structural input features ...")
    clustering = nx.clustering(G)
    core = nx.core_number(G)
    triangles = nx.triangles(G)
    pagerank = nx.pagerank(G)

    rows = []
    for node in node_order:
        rows.append(
            [
                np.log1p(G.degree(node)),
                clustering.get(node, 0.0),
                float(core.get(node, 0)),
                np.log1p(triangles.get(node, 0)),
                pagerank.get(node, 0.0),
            ]
        )
    return np.asarray(rows, dtype=np.float64)

**`main`**

Runs the optional GNN experiment end-to-end: builds the adjacency, trains GCN and GraphSAGE, and reports their accuracy for comparison against the main pipeline's classifiers. It compares two input variants side by side, hand-crafted structural features versus pretrained DeepWalk embeddings, to see whether the message-passing layers add value on top of either input. Not part of the graded course-method results.

In [ ]:
def main(random_states: Sequence[int] = (42, 7, 21, 84, 123)) -> None:
    data_dir = PROJECT_DIR / "data"
    tables_dir = PROJECT_DIR / "outputs" / "tables"
    figures_dir = PROJECT_DIR / "outputs" / "figures"
    embeddings_path = PROJECT_DIR / "outputs" / "embeddings" / "deepwalk_128.npz"
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    G, _, labels_df, _ = load_dataset(data_dir)

    node_order = sorted(int(n) for n in labels_df["id"])
    index_of = {node: idx for idx, node in enumerate(node_order)}
    labels_arr = (
        labels_df.set_index("id").loc[node_order, "label"].to_numpy(dtype=np.int64)
    )

    print("Building normalized adjacencies ...")
    a_hat = build_normalized_adjacency(G, node_order)
    a_mean = build_mean_adjacency(G, node_order)

    # --- Input variant 1: structural features ---
    x_struct = StandardScaler().fit_transform(structural_input_features(G, node_order))

    # --- Input variant 2: DeepWalk embeddings ---
    if not embeddings_path.exists():
        raise FileNotFoundError(
            f"{embeddings_path} not found - run src/run_deepwalk.py first."
        )
    npz = np.load(embeddings_path)
    emb_nodes = npz["nodes"].astype(np.int64)
    emb_matrix = npz["embeddings"].astype(np.float64)
    reorder = np.array([index_of[int(n)] for n in emb_nodes])
    x_deepwalk = np.zeros_like(emb_matrix)
    x_deepwalk[reorder] = emb_matrix
    x_deepwalk = StandardScaler().fit_transform(x_deepwalk)

    results_rows = []
    for run_idx, random_state in enumerate(random_states, start=1):
        print(f"[run {run_idx}/{len(random_states)}] random_state={random_state}")

        train_nodes, test_nodes, y_train, _ = prepare_train_test_split(
            labels_df=labels_df, random_state=int(random_state)
        )
        train_inner_nodes, val_nodes = train_test_split(
            train_nodes, test_size=0.1, random_state=int(random_state), stratify=y_train
        )

        train_idx = np.array([index_of[int(n)] for n in train_inner_nodes])
        val_idx = np.array([index_of[int(n)] for n in val_nodes])
        test_idx = np.array([index_of[int(n)] for n in test_nodes])
        y_test = labels_arr[test_idx]

        for model_name, features, adjacency, architecture in (
            (MODEL_GCN_STRUCT, x_struct, a_hat, "gcn"),
            (MODEL_GCN_DEEPWALK, x_deepwalk, a_hat, "gcn"),
            (MODEL_SAGE_STRUCT, x_struct, a_mean, "graphsage"),
            (MODEL_SAGE_DEEPWALK, x_deepwalk, a_mean, "graphsage"),
        ):
            t0 = time.time()
            y_pred, y_score = train_gcn(
                a_hat=adjacency,
                features=features,
                labels=labels_arr,
                train_idx=train_idx,
                val_idx=val_idx,
                test_idx=test_idx,
                seed=int(random_state),
                architecture=architecture,
            )
            metrics = evaluate_predictions(y_test, y_pred, y_score)
            results_rows.append(
                {
                    "run": run_idx,
                    "random_state": int(random_state),
                    "model": model_name,
                    "accuracy": metrics["accuracy"],
                    "balanced_accuracy": metrics["balanced_accuracy"],
                    "precision": metrics["precision"],
                    "recall": metrics["recall"],
                    "f1": metrics["f1"],
                    "macro_f1": metrics["macro_f1"],
                    "weighted_f1": metrics["weighted_f1"],
                    "roc_auc": metrics["roc_auc"],
                }
            )
            print(
                f"    {model_name}: F1={metrics['f1']:.4f} "
                f"MacroF1={metrics['macro_f1']:.4f} ROC-AUC={metrics['roc_auc']:.4f} "
                f"({time.time() - t0:.0f}s)"
            )

    results_df = pd.DataFrame(results_rows)
    results_df.to_csv(tables_dir / "gnn_model_results.csv", index=False)

    metric_columns = ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "macro_f1", "weighted_f1", "roc_auc"]
    summary_df = results_df.groupby("model", as_index=False)[metric_columns].agg(["mean", "std"])
    summary_df.columns = [
        "model" if col == ("model", "") else f"{col[0]}_{col[1]}" for col in summary_df.columns
    ]
    summary_df.to_csv(tables_dir / "gnn_model_results_summary.csv", index=False)

    # --- Network-only ladder figure ---
    handcrafted_df = pd.read_csv(tables_dir / "repeated_model_results_with_communities.csv")
    deepwalk_df = pd.read_csv(tables_dir / "deepwalk_model_results.csv")

    ladder = [
        ("Handcrafted\n(Graph+Community)",
         handcrafted_df[handcrafted_df["model"] == "Logistic Regression - Graph + Community Features"]),
        ("DeepWalk\n+ LogReg",
         deepwalk_df[deepwalk_df["model"] == "Logistic Regression - DeepWalk Embeddings Only"]),
        ("GCN\n(structural)", results_df[results_df["model"] == MODEL_GCN_STRUCT]),
        ("SAGE\n(structural)", results_df[results_df["model"] == MODEL_SAGE_STRUCT]),
        ("GCN\n(DeepWalk)", results_df[results_df["model"] == MODEL_GCN_DEEPWALK]),
        ("SAGE\n(DeepWalk)", results_df[results_df["model"] == MODEL_SAGE_DEEPWALK]),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, metric, title in zip(axes, ["macro_f1", "roc_auc"], ["Macro F1", "ROC-AUC"]):
        names = [name for name, _ in ladder]
        means = [frame[metric].mean() for _, frame in ladder]
        stds = [frame[metric].std() for _, frame in ladder]
        bars = ax.bar(names, means, yerr=stds, capsize=5,
                      color=["#4f7bd1", "#8a5fc2", "#5aa469", "#8fbc8f", "#2c7a2c", "#1e5e1e"])
        for bar, mean in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width() / 2, mean + 0.008, f"{mean:.3f}",
                    ha="center", va="bottom", fontsize=10)
        ax.set_title(title)
        ax.set_ylim(0.5, 1.0)
        ax.grid(axis="y", alpha=0.3)
    fig.suptitle("Network-Only Ladder: Handcrafted -> Learned -> Message Passing (mean over 5 splits)")
    fig.tight_layout()
    fig.savefig(figures_dir / "gnn_network_only_ladder.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    print("\n=== Summary (mean over splits) ===")
    display_cols = ["model", "f1_mean", "macro_f1_mean", "roc_auc_mean"]
    available = [c for c in display_cols if c in summary_df.columns]
    print(summary_df[available].to_string(index=False))
    print("Saved tables and figure.")

**`__main__ guard`**

Keeps the original command-line entry guard from the source file; it stays inactive here because the run cell below calls the pipeline directly.

In [ ]:
if __name__ == "__main__":
    main()

<h2>Final project wrap-up</h2>

The notebooks show four main points. First, the graph helps predict whether a developer is Web or Machine Learning. Second, content features are very useful, and combining content with graph features helps even more. Third, some hidden edges can be recovered from the graph shape. Fourth, the real graph has stronger same-field linking and community structure than the random comparisons.

<h2>Limits and next steps</h2>

The dataset is a 2019 snapshot, and the label has only two classes. The notebooks show patterns in this dataset, but they do not prove cause and effect. A next version could use time-based edges, richer labels, repository-language features, or the optional extension ideas.

<h2>Summary</h2>

In this notebook we listed the optional extensions and kept them outside the main results. We also wrapped up what the notebooks showed and stated the main limits of the dataset and setup. The goal here is not to add new results, but to make the project ending clear. The final project stays focused, while the extensions remain available as future work.